In [1]:
import os
print(os.getcwd())

c:\Users\aviru\OneDrive\Desktop\MyCodes\python\ML\ExtraProcessing2


In [3]:
import os
import cv2
import numpy as np

# 🔴 IMPORTANT: Paste your FULL dataset path here
input_root = r"C:\Users\aviru\OneDrive\Desktop\MyCodes\python\ML\ExtraProcessing2\sample_Signature"
output_root = r"C:\Users\aviru\OneDrive\Desktop\MyCodes\python\ML\ExtraProcessing2\Processed_extra"

print("Exists:", os.path.exists(input_root))
print("Inside folder:", os.listdir(input_root))

IMG_SIZE = 224

def preprocess_image(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # Binarization (Otsu)
    _, img = cv2.threshold(
        img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    # Noise removal
    kernel = np.ones((3, 3), np.uint8)
    img = cv2.morphologyEx(img, cv2.MORPH_OPEN, kernel)

    # Crop signature area
    coords = cv2.findNonZero(img)
    if coords is None:
        return None

    x, y, w, h = cv2.boundingRect(coords)
    img = img[y:y+h, x:x+w]

    # Resize
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    # Normalize
    img = img / 255.0

    return img


# ✅ Safety Check
print("Dataset exists:", os.path.exists(input_root))
print("Fake exists:", os.path.exists(os.path.join(input_root, "forged")))
print("Real exists:", os.path.exists(os.path.join(input_root, "genuine")))


# 🔄 Process All Images
for category in ["forged", "genuine"]:
    category_path = os.path.join(input_root, category)

    for person_id in os.listdir(category_path):
        person_path = os.path.join(category_path, person_id)

        if not os.path.isdir(person_path):
            continue

        # Create output folder
        save_dir = os.path.join(output_root, category, person_id)
        os.makedirs(save_dir, exist_ok=True)

        for file in os.listdir(person_path):
            img_path = os.path.join(person_path, file)

            processed_img = preprocess_image(img_path)

            if processed_img is not None:
                save_img = (processed_img * 255).astype(np.uint8)
                save_path = os.path.join(save_dir, file)
                cv2.imwrite(save_path, save_img)

print("✅ All images processed and saved successfully!")

Exists: True
Inside folder: ['forged', 'genuine', 'README.txt.txt']
Dataset exists: True
Fake exists: True
Real exists: True
✅ All images processed and saved successfully!
